Schema-unification & dataset merging

Building a unified schema for merging all existing datasets for furter EDA & feature engineering.
- It maps each dataset to a common schema, fills string nulls with "N/A", keeps numerics as real NaNs, and saves a single merged file.

What this does:
- Reads your already-deduped CSVs from prepared/ (no extra file finding).
- Maps each dataset to the unified columns (no country, no source).
- Trims and normalizes text, fills string nulls with "N/A"; numeric salary_numeric stays NaN.
- Concatenates all three into jobs_merged_unified.csv under prepared/.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
%run ../hidden.ipynb

# ----------------------------
# Paths (all from prepared/)
# ----------------------------
p_mr = datasets_folder_path / "prepared" / "morocco_deduped.csv"
p_dc = datasets_folder_path / "prepared" / "dice_com_deduped.csv"
p_sl = datasets_folder_path / "prepared" / "srilanka_deduped.csv"

# ----------------------------
# Unified schema (no country, no source)
# ----------------------------
UNIFIED_COLS = [
    "job_title",
    "role",
    "company",
    "location",
    "employment_type",
    "skills",
    "job_description",
    "responsibilities",
    "qualifications",
    "experience",
    "salary",          # textual range if present
    "salary_numeric",  # numeric if present
    "post_date"
]

def _strip_strings(df: pd.DataFrame) -> pd.DataFrame:
    """Trim whitespace in all object columns."""
    for c in df.select_dtypes(include="object").columns:
        df[c] = df[c].astype(str).str.strip()
        # Normalize empty/placeholder to NaN (so we can fill consistently later)
        df.loc[df[c].isin(["", "nan", "None", "N/A", "NA", "NaN"]), c] = pd.NA
    return df

def _finalize_strings(df: pd.DataFrame) -> pd.DataFrame:
    """Fill missing string columns with 'N/A' (leave numerics as NaN)."""
    obj_cols = df.select_dtypes(include="object").columns.tolist()
    df[obj_cols] = df[obj_cols].fillna("N/A")
    return df

# ----------------------------
# Morocco mapping → unified
# ----------------------------
def load_morocco_unified(path: str | Path): 
    df = pd.read_csv(path, low_memory=False)
    # Expected columns (already deduped and pruned earlier)
    # ['Experience','Qualifications','Salary Range','location','Work Type',
    #  'Job Title','Role','Job Description','skills','Responsibilities',
    #  'Company','Salary_Numeric']
    out = pd.DataFrame({
        "job_title":       df.get("Job Title"),
        "role":            df.get("Role"),
        "company":         df.get("Company"),
        "location":        df.get("location"),
        "employment_type": df.get("Work Type"),
        "skills":          df.get("skills"),
        "job_description": df.get("Job Description"),
        "responsibilities":df.get("Responsibilities"),
        "qualifications":  df.get("Qualifications"),
        "experience":      df.get("Experience"),
        "salary":          df.get("Salary Range"),
        "salary_numeric":  pd.to_numeric(df.get("Salary_Numeric"), errors="coerce"),
        "post_date":       pd.NA,  # not useful/available
    })
    out = _strip_strings(out)
    # Ensure correct column order
    return out.reindex(columns=UNIFIED_COLS)

# ----------------------------
# Dice.com mapping → unified
# ----------------------------
def load_dice_unified(path: str | Path):
    df = pd.read_csv(path, low_memory=False)
    # Expected columns:
    # ['company','employmenttype_jobstatus','jobdescription',
    #  'joblocation_address','jobtitle','postdate','skills']
    out = pd.DataFrame({
        "job_title":       df.get("jobtitle"),
        "role":            pd.NA,  # not present
        "company":         df.get("company"),
        "location":        df.get("joblocation_address"),
        "employment_type": df.get("employmenttype_jobstatus"),
        "skills":          df.get("skills"),
        "job_description": df.get("jobdescription"),
        "responsibilities":pd.NA,
        "qualifications":  pd.NA,
        "experience":      pd.NA,
        "salary":          pd.NA,
        "salary_numeric":  pd.Series([np.nan] * len(df), dtype="float64"),  # Use np.nan instead of pd.NA
        "post_date":       df.get("postdate"),
    })
    out = _strip_strings(out)
    return out.reindex(columns=UNIFIED_COLS)

# ----------------------------
# Sri Lanka mapping → unified
# ----------------------------
def load_srilanka_unified(path: str | Path):
    df = pd.read_csv(path, low_memory=False)
    # Expected columns:
    # ['Job Role','Job Title','Student Qualification','Student Skills','Job Description']
    out = pd.DataFrame({
        "job_title":       df.get("Job Title"),
        "role":            df.get("Job Role"),
        "company":         pd.NA,
        "location":        pd.NA,  # country dropped; location not present
        "employment_type": pd.NA,
        "skills":          df.get("Student Skills"),
        "job_description": df.get("Job Description"),
        "responsibilities":pd.NA,
        "qualifications":  df.get("Student Qualification"),
        "experience":      pd.NA,
        "salary":          pd.NA,
        "salary_numeric":  pd.Series(np.nan, index=df.index, dtype="float64"),
        "post_date":       pd.NA,
    })
    out = _strip_strings(out)
    return out.reindex(columns=UNIFIED_COLS)

# ----------------------------
# Load each → concat → finalize
# ----------------------------
df_mr = load_morocco_unified(p_mr)
df_dc = load_dice_unified(p_dc)
df_sl = load_srilanka_unified(p_sl)

# Merge
df_all = pd.concat([df_mr, df_dc, df_sl], ignore_index=True)

# Fill strings with "N/A" (numeric stays NaN)
df_all = _finalize_strings(df_all)

# Optional: collapse multiple internal spaces & standardize commas in skills/location
def _normalize_commas(s):
    if pd.isna(s): return s
    s = " ".join(str(s).split())        # collapse whitespace
    s = s.replace(" ,", ",").replace(", ", ", ").replace(" , ", ", ")
    return s

for col in ["skills", "location"]:
    if col in df_all.columns:
        df_all[col] = df_all[col].map(_normalize_commas)

print("Unified shape:", df_all.shape)
print("Columns:", df_all.columns.tolist())
print("\nRow counts by source chunk (for reference):")
print("Morocco:", len(df_mr), "| Dice:", len(df_dc), "| SriLanka:", len(df_sl))

# ----------------------------
# Save unified merged dataset
# ----------------------------
out_path = datasets_folder_path / "processed" / "jobs_merged_unified.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df_all.to_csv(out_path, index=False)
print(f"\nSaved merged dataset → processed /jobs_merged_unified.csv")

Unified shape: (40774, 13)
Columns: ['job_title', 'role', 'company', 'location', 'employment_type', 'skills', 'job_description', 'responsibilities', 'qualifications', 'experience', 'salary', 'salary_numeric', 'post_date']

Row counts by source chunk (for reference):
Morocco: 345 | Dice: 20598 | SriLanka: 19831

Saved merged dataset → processed /jobs_merged_unified.csv


📊 Post-Merge Quick Summary

In [10]:
# --- Quick post-merge summary ---

print("=== Unified Dataset Summary ===")
print("Shape:", df_all.shape)

# % of N/A per column
na_pct = (df_all == "N/A").mean().round(3) * 100
print("\n% of 'N/A' values per column:")
print(na_pct.sort_values(ascending=False).to_string())

# Top 10 job titles
print("\nTop 10 Job Titles:")
print(df_all['job_title'].value_counts().head(10).to_string())

# Top 10 employment types
if "employment_type" in df_all.columns:
    print("\nEmployment Type Distribution:")
    print(df_all['employment_type'].value_counts().head(10).to_string())

# Top 10 companies (non-N/A)
print("\nTop 10 Companies:")
print(df_all.loc[df_all['company'] != "N/A", 'company'].value_counts().head(10).to_string())

# Salary summary (numeric only)
if "salary_numeric" in df_all.columns:
    print("\nSalary Numeric (ignoring NaNs):")
    print(df_all['salary_numeric'].describe(percentiles=[.25,.5,.75]))

=== Unified Dataset Summary ===
Shape: (40774, 13)

% of 'N/A' values per column:
employment_type     0.5
company             0.1
skills              0.1
role                0.0
job_title           0.0
location            0.0
job_description     0.0
responsibilities    0.0
qualifications      0.0
experience          0.0
salary              0.0
salary_numeric      0.0
post_date           0.0

Top 10 Job Titles:
job_title
Cloud Infrastructure Engineer    1046
Information Security Analyst     1032
AI Engineer                      1023
System Analyst                   1008
SOC Analyst                      1001
CI/CD Engineer                    995
Help Desk Technician              982
Technical Support Engineer        980
IT Business Analyst               971
NLP Engineer                      961

Employment Type Distribution:
employment_type
<NA>                                                                                                   19831
Full Time                               

Step 3.1 — Employment Type / Role / Title imbalance

Why: Check if categories are balanced enough to be useful as filters, or if too sparse.

What to do:

Count frequencies of employment_type, role, job_title.

See how many unique values there are, and the long tail.

In [11]:
for col in ["employment_type", "role", "job_title"]:
    vc = df_all[col].value_counts(dropna=False)
    print(f"\n=== {col} ===")
    print("Unique:", vc.nunique(), "| Top 10:")
    print(vc.head(10).to_string())


=== employment_type ===
Unique: 83 | Top 10:
employment_type
<NA>                                                                                                   19831
Full Time                                                                                               6264
Contract W2                                                                                             1034
Contract Corp-To-Corp, Contract Independent, Contract W2                                                 593
Full Time, Full Time                                                                                     591
Contract Corp-To-Corp, Contract Independent, Contract W2, C2H Corp-To-Corp, C2H Independent, C2H W2      450
Full Time, Permanent                                                                                     343
Full Time, Full-time, Employee                                                                           321
C2H W2                                                            

Step 3.2 — Company dominance

Why: Some sources (Dice) are dominated by staffing agencies. Knowing this lets you decide whether to treat company as a “weak” feature.

What to do:

Count top 20 companies.

Calculate % of rows that are <NA>.

In [22]:
import numpy as np

vc = df_all["company"].value_counts(dropna=False)

print("\nTop 20 companies:")
print(vc.head(20).to_string())

# Correct way to check NaN values
nan_pct = df_all["company"].isna().mean() * 100
print(f"\n% rows with NaN company: {nan_pct:.2f}%")


Top 20 companies:
company
NaN                         19880
CyberCoders                   325
Amazon                        250
Robert Half Technology        245
Robert Half                   214
Collabera                     166
U.S. Tech Solutions Inc.      163
Kforce Inc.                   161
The Judge Group               157
Visionaire Partners           156
Modis                         137
Apex Systems, Inc             137
Net2Source Inc.               135
Randstad Technologies         131
Deloitte                      123
TEKsystems, Inc.              106
NORTHROP GRUMMAN               95
Experis                        83
iTech Solutions, Inc           75
RSM US                         74

% rows with NaN company: 48.76%


Step 3.3 — Salary coverage

Why: Salary only comes from Morocco (~345 rows). Helps you decide: exclude from MVP or only show if present.

What to do:

Count non-null salary_numeric.

Show distribution.

In [12]:
print("Non-null salary rows:", df_all['salary_numeric'].notna().sum())
print(df_all['salary_numeric'].describe())

Non-null salary rows: 345
count       345.000000
mean     140019.498551
std       40048.567758
min       57656.000000
25%      110702.000000
50%      135222.000000
75%      167853.000000
max      249555.000000
Name: salary_numeric, dtype: float64


Step 3.4 — Skills distribution (most important)

Why: Your recommender is skill-based. You need to know:

Are skills short keywords or messy phrases?

How many unique tokens?

Long tail problem (lots of rare skills)?

What to do:

Split skills strings by delimiters (, ; |) into lists.

Flatten into one big list.

Count frequencies (top 50).

In [13]:
import re
from collections import Counter

SEP_PATTERN = re.compile(r"[,\|;/]+")

def rough_split(s):
    if not isinstance(s, str) or s.strip() in ("", "N/A"): 
        return []
    return [p.strip().lower() for p in SEP_PATTERN.split(s) if p.strip()]

all_skills = [sk for row in df_all["skills"] for sk in rough_split(row)]
freq = Counter(all_skills)

print("Unique raw skills:", len(freq))
print("\nTop 30 skills:")
for skill, count in freq.most_common(30):
    print(f"{skill:<20} {count}")

rare = [s for s, c in freq.items() if c <= 3]
print("\nRare skills (<=3 occurrences):", len(rare))
print(rare)  # print all rare skills

Unique raw skills: 27211

Top 30 skills:
java                 7690
sql                  7615
javascript           7086
linux                6701
c#                   6497
html                 6469
python               6413
css                  6340
excel                6002
aws                  5990
node.js              5807
angular              5793
kubernetes           5672
react                5661
docker               5641
power bi             5601
development          1696
management           1505
agile                964
project              957
testing              887
security             831
analysis             790
oracle               697
.net                 654
unix                 630
windows              625
manager              612
architecture         586
developer            582

Rare skills (<=3 occurrences): 24695
['web accessibility standards (e.g.', 'wcag) assistive technologies accessibility testing tools html and css for accessibility aria (accessible rich inte

Step 3.5 — Job Description vs Skills (spot-check)

Why: Some rows have “skills = N/A”. You can check if job_description contains skill keywords. This tells you if parsing descriptions is worth it.

What to do:

For rows with skills == N/A, print a few job descriptions.

Try matching common skills (Python, SQL, Java, AWS).

In [20]:
# 1. Check how many rows match first:
mask = df_all["skills"] == "N/A"
print("Columns where skills are N/A: ",mask.sum())  # how many rows have skills == "N/A"
# 2. Sample only if non-empty:
subset = df_all[mask]["job_description"]

if not subset.empty:
    n = min(3, len(subset))
    samples = subset.sample(n, random_state=42)
    for i, desc in enumerate(samples, 1):
        print(f"\n=== Sample {i} ===")
        print(desc[:500], "...")
else:
    print("No rows found where skills == 'N/A'")



Columns where skills are N/A:  0
No rows found where skills == 'N/A'


1) Skills cleaning + vocabulary (run once on jobs_merged_unified.csv)

In [15]:
import pandas as pd
import re
from collections import Counter

# Load merged
df_all = pd.read_csv(datasets_folder_path / "processed" / "jobs_merged_unified.csv", low_memory=False)

# --- canonicalization helpers ---
SEP_PATTERN = re.compile(r"[,\|;/]+")
def normalize_skill(s: str) -> str:
    s = s.strip().lower()
    # trivial aliasing
    aliases = {
        "js": "javascript", "node": "node.js", "nodejs": "node.js",
        "py": "python", "postgres": "postgresql",
        "tf": "tensorflow", "sklearn": "scikit-learn"
    }
    return aliases.get(s, s)

def split_skills(cell) -> list[str]:
    if pd.isna(cell) or str(cell).strip() in ("", "N/A"): return []
    # unify delimiters
    parts = SEP_PATTERN.split(str(cell))
    # also split on bullets/whitespace if someone pasted raw text
    out = []
    for p in parts:
        for token in re.split(r"\s*[•·\-\u2022]\s*|\s{2,}", p):
            t = normalize_skill(token)
            if t and len(t) <= 40:  # avoid garbage
                out.append(t)
    # de-dup per row
    return sorted(set(out))

# Clean jobs skills
df_all["skills_list"] = df_all["skills"].apply(split_skills)

# Build vocabulary by frequency (keep useful ones)
freq = Counter([s for row in df_all["skills_list"] for s in row])
vocab = [s for s, c in freq.items() if c >= 5]   # tune threshold if needed
vocab = sorted(vocab)

print("Vocab size:", len(vocab))
print("Examples:", vocab[:30])


Vocab size: 2046
Examples: ['"big data"', '"project manager"', '(help desk or helpdesk or help', '(see job description)', '.net', '.net 4.5', '.net architect', '.net developer', '.net developers', '.net development', '.net framework', '1', '10', '10g', '11g', '12', '12c', '2', '2008', '2010', '2012', '2013', '2014', '2d', '3', '365', '3d', '3gpp', '4', '400']


Dropping Unnecessary Columns